# Qwen3-8B TurboQuant KV-Cache — Performance & Quality Analysis

**Model:** `Qwen/Qwen3-8B` on GKE (NVIDIA GPU)  
**Engine:** vLLM · tensor-parallel 1  
**Load generator:** Locust (streaming SSE)

## Test runs

| Run | Run ID | Experiments | Concurrency |
|-----|--------|-------------|-------------|
| **Run 1** | `run_20260511_234500` | `qwen3_kv_fp8_baseline`, `qwen3_kv_turboquant_4bit_nc`, `qwen3_kv_turboquant_k8v4` | 32 / 64 / 128 |
| **Run 2** | `run_20260512_090311` | `qwen3_kv_turboquant_3bit_nc`, `qwen3_kv_turboquant_k3v4_nc` | 32 / 64 / 128 (3bit), 64 / 128 (k3v4) |

All TurboQuant variants are compared against the **FP8 KV-cache baseline** (`--kv-cache-dtype fp8`).

| Experiment | KV-cache dtype | Notes |
|---|---|---|
| `qwen3_kv_fp8_baseline` | `fp8` | Baseline (8-bit float) |
| `qwen3_kv_turboquant_k8v4` | `turboquant_k8v4` | FP8 keys + 4-bit values, no norm correction |
| `qwen3_kv_turboquant_4bit_nc` | `turboquant_4bit_nc` | 4-bit keys + 4-bit values + norm correction |
| `qwen3_kv_turboquant_k3v4_nc` | `turboquant_k3v4_nc` | 3-bit keys + 4-bit values + norm correction |
| `qwen3_kv_turboquant_3bit_nc` | `turboquant_3bit_nc` | 3-bit keys + 3-bit values + norm correction (max compression) |

## Quality evaluation (`lm-evaluation-harness`)
Two reasoning-mode tasks executed against each KV-cache variant:
- **`arc_challenge`** — 5-shot, 1000 samples — `acc` / `acc_norm`
- **`gsm8k`** — 5-shot, 1000 samples — `exact_match` (strict-match / flexible-extract)

Grafana snapshots:
- Run 1: https://snapshots.raintank.io/dashboard/snapshot/50Lems4r8TZGNzprmjXEGQHuT0rkvsPi
- Run 2: https://snapshots.raintank.io/dashboard/snapshot/dJqN5aINpnl9TmdGr8yUvM4QY0DgpkYN

In [ ]:
import glob, json, os, re, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.size': 10,
    'axes.titlesize': 12,
    'figure.titlesize': 14,
})

# ─── Configuration ──────────────────────────────────────────────────────────
RESULTS_ROOT = Path('results')
EVAL_ROOT    = Path('..') / 'evaluation' / 'results' / 'qwen-3'

RUN1_ID = 'run_20260511_234500'   # fp8_baseline, 4bit_nc, k8v4
RUN2_ID = 'run_20260512_090311'   # 3bit_nc, k3v4_nc

BASELINE_EXP = 'qwen3_kv_fp8_baseline'

# Order experiments from least to most aggressive KV compression
EXP_ORDER = [
    'qwen3_kv_fp8_baseline',
    'qwen3_kv_turboquant_k8v4',
    'qwen3_kv_turboquant_4bit_nc',
    'qwen3_kv_turboquant_k3v4_nc',
    'qwen3_kv_turboquant_3bit_nc',
]
EXP_SHORT = {
    'qwen3_kv_fp8_baseline':       'fp8 (baseline)',
    'qwen3_kv_turboquant_k8v4':    'tq k8v4',
    'qwen3_kv_turboquant_4bit_nc': 'tq 4bit_nc',
    'qwen3_kv_turboquant_k3v4_nc': 'tq k3v4_nc',
    'qwen3_kv_turboquant_3bit_nc': 'tq 3bit_nc',
}

# Mapping: benchmark experiment name -> evaluation results subfolder
EXP_TO_EVAL_DIR = {
    'qwen3_kv_fp8_baseline':       'fp8-model&fp8-kv-cache',
    'qwen3_kv_turboquant_k8v4':    'fp8-model&turboquant_k8v4-kv-cache',
    'qwen3_kv_turboquant_4bit_nc': 'fp8-model&turboquant_4bit_nc-kv-cache',
    'qwen3_kv_turboquant_k3v4_nc': 'fp8-model&turboquant_k3v4_nc-kv-cache',
    'qwen3_kv_turboquant_3bit_nc': 'fp8-model&turboquant_3bit_nc-kv-cache',
}

GPU_HOURLY_COST_USD = 1.58

## 1. Load benchmark data (Locust + Prometheus)

In [ ]:
def load_run_locust(run_dir: Path, run_label: str) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(run_dir.rglob('*custom_metrics*.csv')):
        stem = csv_path.stem
        exp  = csv_path.parent.name
        m = re.search(r'__u(\d+)', stem)
        users = int(m.group(1)) if m else 0
        df = pd.read_csv(csv_path)
        df['experiment'], df['users'], df['run'] = exp, users, run_label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_run_prometheus(run_dir: Path, run_label: str) -> pd.DataFrame:
    frames = []
    for csv_path in sorted(run_dir.rglob('*prometheus_metrics*.csv')):
        stem = csv_path.stem
        exp  = csv_path.parent.name
        m = re.search(r'__u(\d+)', stem)
        users = int(m.group(1)) if m else 0
        try:
            df = pd.read_csv(csv_path)
            df['experiment'], df['users'], df['run'] = exp, users, run_label
            frames.append(df)
        except Exception as e:
            print(f'Warning: {csv_path.name}: {e}')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

run1_dir = RESULTS_ROOT / RUN1_ID
run2_dir = RESULTS_ROOT / RUN2_ID

df_run1 = load_run_locust(run1_dir, 'run1')
df_run2 = load_run_locust(run2_dir, 'run2')
df_all  = pd.concat([df_run1, df_run2], ignore_index=True)
df_ok   = df_all[df_all['success'] == True].copy()

prom_run1 = load_run_prometheus(run1_dir, 'run1')
prom_run2 = load_run_prometheus(run2_dir, 'run2')
prom_all  = pd.concat([prom_run1, prom_run2], ignore_index=True)

# Drop experiments that have no data (e.g. empty k3v4_nc folder in Run 1)
df_ok = df_ok[df_ok['experiment'].isin(EXP_ORDER)].copy()

print(f'Run 1  ({RUN1_ID}): {len(df_run1):>6,} rows | {df_run1["experiment"].nunique()} experiments')
print(f'Run 2  ({RUN2_ID}): {len(df_run2):>6,} rows | {df_run2["experiment"].nunique()} experiments')
print('────────────────────────────────────────────')
print(f'Total loaded   : {len(df_all):>6,}')
print(f'Successful     : {len(df_ok):>6,} ({100*len(df_ok)/max(len(df_all),1):.1f}%)')
print(f'Failed         : {len(df_all)-len(df_ok):>6,}')
print(f'\nPrometheus metrics: {len(prom_all):,} rows')

df_ok.head(3)

In [ ]:
# ─── Experiment matrix ─────────────────────────────────────────────────────
matrix = (
    df_ok.groupby(['experiment', 'users'])
    .agg(n=('ttft_ms', 'count'))
    .reset_index()
    .pivot_table(index='experiment', columns='users', values='n', fill_value=0)
    .astype(int)
)
matrix.columns = [f'u{c}' for c in matrix.columns]
matrix = matrix.reindex([e for e in EXP_ORDER if e in matrix.index])

# Pull short description from each experiment's config.json
desc_map = {}
for cfg in RESULTS_ROOT.rglob('config.json'):
    c = json.loads(cfg.read_text())
    desc_map[c['name']] = c.get('description', '')
matrix['description'] = [desc_map.get(e, '') for e in matrix.index]

print('Experiment matrix (successful request counts per concurrency):')
display(matrix)

## 2. Client-side latency summary

In [ ]:
def pct(s, p):
    return s.quantile(p / 100)

def build_summary(df, by_category=False):
    group_cols = ['experiment', 'users']
    if by_category and 'category' in df.columns:
        group_cols.append('category')
    return (
        df.groupby(group_cols)
        .agg(
            n             = ('ttft_ms', 'count'),
            ttft_mean     = ('ttft_ms', 'mean'),
            ttft_p50      = ('ttft_ms', lambda s: pct(s, 50)),
            ttft_p95      = ('ttft_ms', lambda s: pct(s, 95)),
            ttft_p99      = ('ttft_ms', lambda s: pct(s, 99)),
            tpot_mean     = ('tpot_ms', 'mean'),
            tpot_p50      = ('tpot_ms', lambda s: pct(s, 50)),
            tpot_p99      = ('tpot_ms', lambda s: pct(s, 99)),
            e2e_mean      = ('e2e_ms', 'mean'),
            e2e_p50       = ('e2e_ms', lambda s: pct(s, 50)),
            e2e_p99       = ('e2e_ms', lambda s: pct(s, 99)),
            itl_p50_mean  = ('itl_p50_ms', 'mean'),
            itl_p99_mean  = ('itl_p99_ms', 'mean'),
            output_tokens = ('output_tokens', 'sum'),
        )
        .reset_index()
    )

summary_agg = build_summary(df_ok)
summary_cat = build_summary(df_ok, by_category=True)

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.1f}'.format)

print('Aggregated latency summary (across all prompt categories):')
display(summary_agg.set_index(['experiment', 'users']).reindex(
    [(e, u) for e in EXP_ORDER for u in sorted(summary_agg['users'].unique())]
).dropna(how='all'))

## 3. Latency distributions (TTFT / TPOT / E2E)

In [ ]:
def _exp_order_in(df):
    return [e for e in EXP_ORDER if e in df['experiment'].unique()]

def violin_by_users(df, metric, title, ylabel, palette='Set2', fmt='{:.0f}'):
    user_levels = sorted(df['users'].unique())
    n_u = len(user_levels)
    fig, axes = plt.subplots(1, n_u, figsize=(5.5 * n_u, 6), sharey=False)
    if n_u == 1:
        axes = [axes]
    for ax, u in zip(axes, user_levels):
        sub = df[df['users'] == u]
        order = [e for e in _exp_order_in(sub)]
        sns.violinplot(data=sub, x='experiment', y=metric, order=order,
                       inner='box', cut=0, ax=ax, palette=palette, alpha=0.75)
        ax.set_title(f'{title} — {u} user(s)', fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel(ylabel)
        ax.set_xticklabels([EXP_SHORT.get(e, e) for e in order], rotation=25, ha='right')
        for i, exp in enumerate(order):
            med = sub[sub['experiment'] == exp][metric].median()
            if pd.notna(med):
                ax.annotate(fmt.format(med), (i, med), textcoords='offset points',
                            xytext=(0, -12), ha='center', fontsize=7, fontweight='bold')
    plt.suptitle(f'{title} Distribution — FP8 baseline vs TurboQuant', fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig = violin_by_users(df_ok, 'ttft_ms', 'TTFT', 'TTFT (ms)', palette='Set2', fmt='{:.0f}')
plt.savefig(RESULTS_ROOT / 'turboquant_ttft_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
fig = violin_by_users(df_ok, 'tpot_ms', 'TPOT', 'TPOT (ms/token)', palette='Pastel1', fmt='{:.1f}')
plt.savefig(RESULTS_ROOT / 'turboquant_tpot_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── E2E latency: boxplot is cleaner for long tails ────────────────────────
user_levels = sorted(df_ok['users'].unique())
n_u = len(user_levels)
fig, axes = plt.subplots(1, n_u, figsize=(5.5 * n_u, 6), sharey=False)
if n_u == 1:
    axes = [axes]
for ax, u in zip(axes, user_levels):
    sub = df_ok[df_ok['users'] == u]
    order = [e for e in _exp_order_in(sub)]
    sns.boxplot(data=sub, x='experiment', y='e2e_ms', order=order,
                ax=ax, palette='Set3',
                flierprops=dict(marker='.', markersize=3, alpha=0.4))
    ax.set_title(f'E2E — {u} user(s)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('E2E Latency (ms)')
    ax.set_xticklabels([EXP_SHORT.get(e, e) for e in order], rotation=25, ha='right')
plt.suptitle('End-to-End Latency Distribution — FP8 baseline vs TurboQuant', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_e2e_boxplot.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Inter-token latency (ITL) — P50 and P99 ───────────────────────────────
itl_agg = (
    df_ok.groupby(['experiment', 'users'])
    .agg(itl_p50=('itl_p50_ms', 'mean'), itl_p99=('itl_p99_ms', 'mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, metric, title in zip(axes, ['itl_p50', 'itl_p99'], ['ITL P50', 'ITL P99']):
    pivot = itl_agg.pivot(index='experiment', columns='users', values=metric)
    pivot = pivot.reindex([e for e in EXP_ORDER if e in pivot.index])
    pivot.index = [EXP_SHORT.get(e, e) for e in pivot.index]
    pivot.plot(kind='barh', ax=ax, width=0.75)
    ax.set_xlabel('Inter-Token Latency (ms)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('')
    ax.legend(title='Users', fontsize=8)

plt.suptitle('Inter-Token Latency — FP8 baseline vs TurboQuant', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_itl_analysis.png', bbox_inches='tight')
plt.show()

## 4. Throughput vs latency — Pareto frontier

In [ ]:
def compute_throughput(grp):
    total_tokens = grp['output_tokens'].sum()
    duration_s = grp['timestamp'].max() - grp['timestamp'].min()
    if duration_s < 1:
        duration_s = 1
    return pd.Series({
        'token_throughput': total_tokens / duration_s,
        'ttft_p95':         grp['ttft_ms'].quantile(0.95),
        'ttft_p50':         grp['ttft_ms'].quantile(0.50),
        'e2e_p95':          grp['e2e_ms'].quantile(0.95),
    })

pareto = df_ok.groupby(['experiment', 'users']).apply(compute_throughput).reset_index()

fig, ax = plt.subplots(figsize=(11, 7))
palette = sns.color_palette('tab10', len(EXP_ORDER))

for exp, color in zip([e for e in EXP_ORDER if e in pareto['experiment'].unique()], palette):
    sub = pareto[pareto['experiment'] == exp].sort_values('users')
    ax.plot(sub['ttft_p95'], sub['token_throughput'],
            marker='o', label=EXP_SHORT.get(exp, exp), color=color, linewidth=2, markersize=7)
    for _, row in sub.iterrows():
        ax.annotate(f'u={int(row["users"])}', (row['ttft_p95'], row['token_throughput']),
                    textcoords='offset points', xytext=(5, 5), fontsize=7, color=color)

ax.set_xlabel('P95 TTFT (ms)  ← lower is better')
ax.set_ylabel('Output Token Throughput (tok/s)  ↑ higher is better')
ax.set_title('Throughput vs P95 TTFT — FP8 baseline vs TurboQuant', fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_pareto_frontier.png', bbox_inches='tight')
plt.show()

## 5. Latency heatmaps

In [ ]:
metrics_for_heatmap = [
    ('ttft_p50', 'P50 TTFT (ms)',     'RdYlGn_r'),
    ('ttft_p99', 'P99 TTFT (ms)',     'RdYlGn_r'),
    ('tpot_p50', 'P50 TPOT (ms/tok)', 'RdYlGn_r'),
    ('e2e_p50',  'P50 E2E (ms)',      'RdYlGn_r'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, title, cmap) in zip(axes.flat, metrics_for_heatmap):
    pivot = summary_agg.pivot_table(index='experiment', columns='users', values=col, aggfunc='mean')
    pivot = pivot.reindex([e for e in EXP_ORDER if e in pivot.index])
    pivot.index = [EXP_SHORT.get(e, e) for e in pivot.index]
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap=cmap,
                linewidths=0.5, ax=ax, cbar_kws={'label': title})
    ax.set_title(title + ' — lower is better', fontweight='bold', fontsize=11)
    ax.set_xlabel('Concurrent Users')
    ax.set_ylabel('')

plt.suptitle('Latency Heatmaps — FP8 baseline vs TurboQuant', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_latency_heatmaps.png', bbox_inches='tight')
plt.show()

## 6. Percentage change vs FP8 baseline

In [ ]:
base = summary_agg[summary_agg['experiment'] == BASELINE_EXP].set_index('users')
rows = []
for exp in [e for e in EXP_ORDER if e != BASELINE_EXP and e in summary_agg['experiment'].unique()]:
    comp = summary_agg[summary_agg['experiment'] == exp].set_index('users')
    for u in comp.index.intersection(base.index):
        b, c = base.loc[u], comp.loc[u]
        rows.append({
            'experiment':         exp,
            'users':              u,
            'ttft_p50_change_%':  (c['ttft_p50'] - b['ttft_p50']) / b['ttft_p50'] * 100,
            'ttft_p99_change_%':  (c['ttft_p99'] - b['ttft_p99']) / b['ttft_p99'] * 100,
            'tpot_p50_change_%':  (c['tpot_p50'] - b['tpot_p50']) / b['tpot_p50'] * 100,
            'tpot_p99_change_%':  (c['tpot_p99'] - b['tpot_p99']) / b['tpot_p99'] * 100,
            'e2e_p50_change_%':   (c['e2e_p50']  - b['e2e_p50'])  / b['e2e_p50']  * 100,
            'e2e_p99_change_%':   (c['e2e_p99']  - b['e2e_p99'])  / b['e2e_p99']  * 100,
        })

imp_df = pd.DataFrame(rows)
print(f'Percentage change vs {BASELINE_EXP} (negative = improvement, positive = regression):')
display(imp_df.style.format({c: '{:+.1f}%' for c in imp_df.columns if c.endswith('_change_%')})
        .map(lambda v: 'color: green' if isinstance(v, str) and v.startswith('-')
             else ('color: red' if isinstance(v, str) and v.startswith('+') else ''),
             subset=[c for c in imp_df.columns if c.endswith('_change_%')]))

# ─── Bar chart per metric ──────────────────────────────────────────────────
metrics_to_plot = ['ttft_p50_change_%', 'tpot_p50_change_%', 'e2e_p50_change_%']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 5), sharey=False)
for ax, metric in zip(axes, metrics_to_plot):
    pivot = imp_df.pivot(index='experiment', columns='users', values=metric)
    pivot = pivot.reindex([e for e in EXP_ORDER if e in pivot.index])
    pivot.index = [EXP_SHORT.get(e, e) for e in pivot.index]
    pivot.plot(kind='bar', ax=ax, width=0.75, colormap='coolwarm')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel(f'{metric.replace("_change_%", "").upper()} change (%)')
    ax.set_title(f'{metric.replace("_change_%", "").upper()} vs FP8 baseline', fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='Users', fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_pct_change_vs_baseline.png', bbox_inches='tight')
plt.show()

## 7. Cost per million output tokens

In [ ]:
cost_rows = []
for (exp, users), grp in df_ok.groupby(['experiment', 'users']):
    total_tokens = grp['output_tokens'].sum()
    duration_hr  = (grp['timestamp'].max() - grp['timestamp'].min()) / 3600
    if duration_hr < 1e-6 or total_tokens == 0:
        continue
    cost_per_m = (GPU_HOURLY_COST_USD * duration_hr) / (total_tokens / 1_000_000)
    cost_rows.append({
        'experiment':        exp,
        'users':             users,
        'total_tokens':      total_tokens,
        'duration_s':        duration_hr * 3600,
        'cost_per_M_tokens': cost_per_m,
    })

cost_df = pd.DataFrame(cost_rows)
cost_df['experiment'] = pd.Categorical(cost_df['experiment'], categories=EXP_ORDER, ordered=True)
cost_df = cost_df.sort_values(['experiment', 'users'])

fig, ax = plt.subplots(figsize=(11, 5.5))
for exp in [e for e in EXP_ORDER if e in cost_df['experiment'].unique()]:
    sub = cost_df[cost_df['experiment'] == exp]
    ax.plot(sub['users'], sub['cost_per_M_tokens'], marker='o',
            label=EXP_SHORT.get(exp, exp), linewidth=1.8)

ax.set_xlabel('Concurrent Users')
ax.set_ylabel('USD per Million Output Tokens')
ax.set_title(f'Cost Efficiency — GPU rate: ${GPU_HOURLY_COST_USD}/hr', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_cost_per_token.png', bbox_inches='tight')
plt.show()

display(cost_df.style.format({
    'cost_per_M_tokens': '${:.2f}',
    'duration_s':        '{:.0f}s',
    'total_tokens':      '{:,}',
}))

## 8. Server-side metrics (Prometheus)

GPU utilization, VRAM, KV-cache pressure, queue depth and generation throughput as observed on the vLLM pod during each (experiment, concurrency) cell.

In [ ]:
def pivot_prometheus(prom_df: pd.DataFrame) -> pd.DataFrame:
    if prom_df.empty:
        return pd.DataFrame()
    pdf = prom_df.copy()
    pdf['value'] = pd.to_numeric(pdf['value'], errors='coerce')
    wide = pdf.pivot_table(
        index=['experiment', 'users', 'run', 'timestamp'],
        columns='metric_name',
        values='value',
        aggfunc='first',
    ).reset_index()
    wide.columns.name = None
    return wide

prom_wide = pivot_prometheus(prom_all)
prom_wide = prom_wide[prom_wide['experiment'].isin(EXP_ORDER)].copy()
if not prom_wide.empty:
    prom_wide['elapsed_s'] = prom_wide.groupby(['experiment', 'users'])['timestamp'].transform(
        lambda x: x - x.min()
    )

print(f'Prometheus wide: {len(prom_wide):,} rows')
available = [c for c in prom_wide.columns if c not in ['experiment','users','run','timestamp','elapsed_s']]
print(f'Available metrics: {available}')

In [ ]:
# ─── Server-side aggregate summary ─────────────────────────────────────────
agg_cols = {}
if 'gpu_util_pct'              in prom_wide.columns: agg_cols['gpu_util_mean']         = ('gpu_util_pct', 'mean')
if 'gpu_util_pct'              in prom_wide.columns: agg_cols['gpu_util_max']          = ('gpu_util_pct', 'max')
if 'gpu_fb_used_mib'           in prom_wide.columns: agg_cols['vram_mean_mib']         = ('gpu_fb_used_mib', 'mean')
if 'gpu_fb_used_mib'           in prom_wide.columns: agg_cols['vram_max_mib']          = ('gpu_fb_used_mib', 'max')
if 'kv_cache_usage_pct'        in prom_wide.columns: agg_cols['kv_cache_mean_pct']     = ('kv_cache_usage_pct', 'mean')
if 'kv_cache_usage_pct'        in prom_wide.columns: agg_cols['kv_cache_max_pct']      = ('kv_cache_usage_pct', 'max')
if 'requests_running'          in prom_wide.columns: agg_cols['requests_running_mean'] = ('requests_running', 'mean')
if 'requests_waiting'          in prom_wide.columns: agg_cols['requests_waiting_mean'] = ('requests_waiting', 'mean')
if 'requests_waiting'          in prom_wide.columns: agg_cols['requests_waiting_max']  = ('requests_waiting', 'max')
if 'generation_tokens_per_sec' in prom_wide.columns: agg_cols['gen_throughput_mean']   = ('generation_tokens_per_sec', 'mean')

server_summary = (
    prom_wide.groupby(['experiment', 'users'])
    .agg(**agg_cols)
    .reset_index()
)
server_summary['experiment'] = pd.Categorical(server_summary['experiment'], categories=EXP_ORDER, ordered=True)
server_summary = server_summary.sort_values(['experiment', 'users']).reset_index(drop=True)

pd.set_option('display.float_format', '{:.1f}'.format)
print('Server-side summary:')
display(server_summary)

In [ ]:
# ─── KV cache usage heatmap ────────────────────────────────────────────────
if 'kv_cache_mean_pct' in server_summary.columns:
    pivot_kv = server_summary.pivot_table(
        index='experiment', columns='users', values='kv_cache_mean_pct', aggfunc='mean'
    )
    pivot_kv = pivot_kv.reindex([e for e in EXP_ORDER if e in pivot_kv.index])
    pivot_kv.index = [EXP_SHORT.get(e, e) for e in pivot_kv.index]
    fig, ax = plt.subplots(figsize=(max(8, pivot_kv.shape[1] * 2), max(4, pivot_kv.shape[0] * 0.7)))
    sns.heatmap(pivot_kv, annot=True, fmt='.1f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'KV Cache Usage (%)'})
    ax.set_title('Mean KV-Cache Utilization (%) — higher = more memory pressure', fontweight='bold')
    ax.set_xlabel('Concurrent Users')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'turboquant_kv_cache_heatmap.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ─── GPU util + Generation throughput bars ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if 'gpu_util_mean' in server_summary.columns:
    p = server_summary.pivot(index='experiment', columns='users', values='gpu_util_mean')
    p = p.reindex([e for e in EXP_ORDER if e in p.index])
    p.index = [EXP_SHORT.get(e, e) for e in p.index]
    p.plot(kind='bar', ax=axes[0], width=0.75)
    axes[0].set_ylabel('Mean GPU Utilization (%)')
    axes[0].set_ylim(0, 105)
    axes[0].set_xlabel('')
    axes[0].set_title('Mean GPU Utilization', fontweight='bold')
    axes[0].legend(title='Users', fontsize=8)
    axes[0].tick_params(axis='x', rotation=20)

if 'gen_throughput_mean' in server_summary.columns:
    p = server_summary.pivot(index='experiment', columns='users', values='gen_throughput_mean')
    p = p.reindex([e for e in EXP_ORDER if e in p.index])
    p.index = [EXP_SHORT.get(e, e) for e in p.index]
    p.plot(kind='bar', ax=axes[1], width=0.75, colormap='viridis')
    axes[1].set_ylabel('Avg Generation Throughput (tok/s)')
    axes[1].set_xlabel('')
    axes[1].set_title('Generation Throughput (vLLM)', fontweight='bold')
    axes[1].legend(title='Users', fontsize=8)
    axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Server-side: GPU Utilization & Generation Throughput', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_server_bars.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── VRAM comparison (mean and max) ────────────────────────────────────────
if 'vram_mean_mib' in server_summary.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, col, title in zip(axes, ['vram_mean_mib', 'vram_max_mib'], ['Mean VRAM (MiB)', 'Max VRAM (MiB)']):
        p = server_summary.pivot(index='experiment', columns='users', values=col)
        p = p.reindex([e for e in EXP_ORDER if e in p.index])
        p.index = [EXP_SHORT.get(e, e) for e in p.index]
        p.plot(kind='bar', ax=ax, width=0.75, colormap='Reds')
        ax.set_ylabel('VRAM (MiB)')
        ax.set_xlabel('')
        ax.set_title(title, fontweight='bold')
        ax.legend(title='Users', fontsize=8)
        ax.tick_params(axis='x', rotation=20)
    plt.suptitle('VRAM Usage — FP8 baseline vs TurboQuant', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'turboquant_vram.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ─── KV-cache & queue depth time series — one panel per (experiment, users) ─
if not prom_wide.empty and 'kv_cache_usage_pct' in prom_wide.columns:
    combos = prom_wide.groupby(['experiment', 'users']).ngroups
    n_cols = min(4, combos)
    n_rows = (combos + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.6 * n_rows), squeeze=False)
    axes_flat = axes.flatten()

    ordered_groups = sorted(
        prom_wide.groupby(['experiment', 'users']),
        key=lambda kv: (EXP_ORDER.index(kv[0][0]) if kv[0][0] in EXP_ORDER else 99, kv[0][1]),
    )

    for idx, ((exp, u), grp) in enumerate(ordered_groups):
        if idx >= len(axes_flat):
            break
        ax = axes_flat[idx]
        ax2 = ax.twinx()
        ax.plot(grp['elapsed_s'], grp['kv_cache_usage_pct'], color='tab:green', alpha=0.85, label='KV %')
        if 'requests_running' in grp.columns:
            ax2.plot(grp['elapsed_s'], grp['requests_running'], color='tab:orange', alpha=0.85, label='Running')
        if 'requests_waiting' in grp.columns:
            ax2.plot(grp['elapsed_s'], grp['requests_waiting'], color='tab:red', alpha=0.6, linestyle='--', label='Waiting')
        ax.set_ylim(0, 105)
        ax.set_ylabel('KV %', color='tab:green', fontsize=8)
        ax2.set_ylabel('Req count', color='tab:orange', fontsize=8)
        ax.set_xlabel('Elapsed (s)')
        ax.set_title(f'{EXP_SHORT.get(exp, exp)} — u{u}', fontsize=9)
        ax.legend(loc='upper left', fontsize=6)
        ax2.legend(loc='upper right', fontsize=6)

    for j in range(idx + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle('KV-Cache Usage & Queue Depth — Time Series', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / 'turboquant_kv_queue_timeseries.png', bbox_inches='tight')
    plt.show()

## 9. Quality evaluation — `arc_challenge` & `gsm8k`

Run via `lm-evaluation-harness` against each vLLM endpoint. Each task uses 1000 samples, 5-shot, with reasoning (`enable_thinking=true`) enabled.

In [ ]:
def load_eval_results():
    """Walk EVAL_ROOT/<variant>/reasoning/<task>/results_*.json and collect headline metrics."""
    rows = []
    for exp, eval_dir in EXP_TO_EVAL_DIR.items():
        base = EVAL_ROOT / eval_dir / 'reasoning'
        if not base.exists():
            print(f'⚠ Missing eval dir: {base}')
            continue
        for task_dir in base.iterdir():
            if not task_dir.is_dir():
                continue
            task = task_dir.name
            if task not in ('arc_challenge', 'gsm8k'):
                continue  # skip legacy mmlu rows present only for the FP8 baseline
            for jf in sorted(task_dir.glob('results_*.json')):
                d = json.loads(jf.read_text())
                res  = d.get('results', {}).get(task, {})
                row = {
                    'experiment':   exp,
                    'task':         task,
                    'samples':      res.get('sample_len'),
                    'timestamp':    jf.stem.replace('results_', ''),
                }
                if task == 'arc_challenge':
                    row['acc']             = res.get('acc,none')
                    row['acc_stderr']      = res.get('acc_stderr,none')
                    row['acc_norm']        = res.get('acc_norm,none')
                    row['acc_norm_stderr'] = res.get('acc_norm_stderr,none')
                elif task == 'gsm8k':
                    row['exact_match_strict']    = res.get('exact_match,strict-match')
                    row['exact_match_strict_se'] = res.get('exact_match_stderr,strict-match')
                    row['exact_match_flex']      = res.get('exact_match,flexible-extract')
                    row['exact_match_flex_se']   = res.get('exact_match_stderr,flexible-extract')
                rows.append(row)
    return pd.DataFrame(rows)

eval_df = load_eval_results()
eval_df['experiment'] = pd.Categorical(eval_df['experiment'], categories=EXP_ORDER, ordered=True)
eval_df = eval_df.sort_values(['task', 'experiment']).reset_index(drop=True)

print('Quality evaluation raw rows:')
display(eval_df)


In [ ]:
# ─── ARC Challenge accuracy ────────────────────────────────────────────────
arc = eval_df[eval_df['task'] == 'arc_challenge'].copy()
gsm = eval_df[eval_df['task'] == 'gsm8k'].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ARC
ax = axes[0]
x = np.arange(len(arc))
w = 0.35
ax.bar(x - w/2, arc['acc'],      width=w, yerr=arc['acc_stderr'],      capsize=4,
       label='acc',      color='steelblue', alpha=0.9)
ax.bar(x + w/2, arc['acc_norm'], width=w, yerr=arc['acc_norm_stderr'], capsize=4,
       label='acc_norm', color='orange',    alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([EXP_SHORT.get(e, e) for e in arc['experiment']], rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('ARC-Challenge — 5-shot, 1000 samples', fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend()
for i, v in enumerate(arc['acc']):
    ax.annotate(f'{v:.3f}', (i - w/2, v), textcoords='offset points', xytext=(0, 4), ha='center', fontsize=8)
for i, v in enumerate(arc['acc_norm']):
    ax.annotate(f'{v:.3f}', (i + w/2, v), textcoords='offset points', xytext=(0, 4), ha='center', fontsize=8)

# GSM8K
ax = axes[1]
x = np.arange(len(gsm))
ax.bar(x - w/2, gsm['exact_match_strict'], width=w, yerr=gsm['exact_match_strict_se'], capsize=4,
       label='strict-match',     color='seagreen',  alpha=0.9)
ax.bar(x + w/2, gsm['exact_match_flex'],   width=w, yerr=gsm['exact_match_flex_se'],   capsize=4,
       label='flexible-extract', color='mediumpurple', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([EXP_SHORT.get(e, e) for e in gsm['experiment']], rotation=20, ha='right')
ax.set_ylabel('Exact-match accuracy')
ax.set_title('GSM8K — 5-shot, 1000 samples', fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend()
for i, v in enumerate(gsm['exact_match_strict']):
    ax.annotate(f'{v:.3f}', (i - w/2, v), textcoords='offset points', xytext=(0, 4), ha='center', fontsize=8)
for i, v in enumerate(gsm['exact_match_flex']):
    ax.annotate(f'{v:.3f}', (i + w/2, v), textcoords='offset points', xytext=(0, 4), ha='center', fontsize=8)

plt.suptitle('Quality — FP8 baseline vs TurboQuant', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_quality_bars.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Accuracy drop vs FP8 baseline ─────────────────────────────────────────
base_arc = arc[arc['experiment'] == BASELINE_EXP].iloc[0]
base_gsm = gsm[gsm['experiment'] == BASELINE_EXP].iloc[0]

quality_diff = []
for exp in [e for e in EXP_ORDER if e != BASELINE_EXP]:
    a = arc[arc['experiment'] == exp]
    g = gsm[gsm['experiment'] == exp]
    if a.empty or g.empty:
        continue
    a = a.iloc[0]; g = g.iloc[0]
    quality_diff.append({
        'experiment':            exp,
        'arc_acc':               a['acc'],
        'arc_acc_Δ':             a['acc'] - base_arc['acc'],
        'arc_acc_norm':          a['acc_norm'],
        'arc_acc_norm_Δ':        a['acc_norm'] - base_arc['acc_norm'],
        'gsm8k_strict':          g['exact_match_strict'],
        'gsm8k_strict_Δ':        g['exact_match_strict'] - base_gsm['exact_match_strict'],
        'gsm8k_flex':            g['exact_match_flex'],
        'gsm8k_flex_Δ':          g['exact_match_flex']   - base_gsm['exact_match_flex'],
    })

quality_diff_df = pd.DataFrame(quality_diff)
print(f'Accuracy delta vs {BASELINE_EXP} (positive = better than baseline):')
display(quality_diff_df.style.format({
    'arc_acc': '{:.3f}', 'arc_acc_norm': '{:.3f}',
    'gsm8k_strict': '{:.3f}', 'gsm8k_flex': '{:.3f}',
    'arc_acc_Δ': '{:+.3f}', 'arc_acc_norm_Δ': '{:+.3f}',
    'gsm8k_strict_Δ': '{:+.3f}', 'gsm8k_flex_Δ': '{:+.3f}',
}))

## 10. Quality vs performance trade-off

For each KV-cache variant, plot accuracy on each task against headline performance metrics aggregated across all concurrency levels. This is the punch-line of the analysis: does aggressive KV-cache compression actually pay off?

In [ ]:
# Aggregate per experiment across all concurrency levels
perf_summary = (
    df_ok.groupby('experiment')
    .agg(
        ttft_p50  = ('ttft_ms', lambda s: s.quantile(0.5)),
        ttft_p95  = ('ttft_ms', lambda s: s.quantile(0.95)),
        tpot_p50  = ('tpot_ms', lambda s: s.quantile(0.5)),
        e2e_p50   = ('e2e_ms',  lambda s: s.quantile(0.5)),
    )
    .reset_index()
)

# Throughput at the highest shared concurrency (128 users)
tput_u128 = pareto[pareto['users'] == 128][['experiment', 'token_throughput']].rename(
    columns={'token_throughput': 'throughput_u128_tok_s'}
)
perf_summary = perf_summary.merge(tput_u128, on='experiment', how='left')

# Server-side: KV-cache and VRAM at the highest concurrency
server_u128 = server_summary[server_summary['users'] == 128][
    ['experiment'] + [c for c in ['kv_cache_mean_pct', 'vram_mean_mib', 'gen_throughput_mean']
                       if c in server_summary.columns]
]
perf_summary = perf_summary.merge(server_u128, on='experiment', how='left')

# Attach quality scores
qmap = (
    arc[['experiment', 'acc', 'acc_norm']]
    .rename(columns={'acc': 'arc_acc', 'acc_norm': 'arc_acc_norm'})
    .merge(
        gsm[['experiment', 'exact_match_strict', 'exact_match_flex']]
        .rename(columns={'exact_match_strict': 'gsm8k_strict', 'exact_match_flex': 'gsm8k_flex'}),
        on='experiment', how='outer',
    )
)
perf_summary = perf_summary.merge(qmap, on='experiment', how='left')
perf_summary['experiment'] = pd.Categorical(perf_summary['experiment'], categories=EXP_ORDER, ordered=True)
perf_summary = perf_summary.sort_values('experiment').reset_index(drop=True)

print('Per-experiment performance × quality summary:')
display(perf_summary)

In [ ]:
# ─── Quality vs Throughput scatter ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
palette = sns.color_palette('tab10', len(EXP_ORDER))
color_map = {e: c for e, c in zip(EXP_ORDER, palette)}

for ax, qmetric, qlabel, ylim in zip(
    axes,
    ['arc_acc', 'gsm8k_strict'],
    ['ARC-Challenge acc', 'GSM8K exact-match (strict)'],
    [(0.55, 0.70), (0.80, 0.92)],
):
    for _, r in perf_summary.iterrows():
        if pd.isna(r['throughput_u128_tok_s']) or pd.isna(r[qmetric]):
            continue
        c = color_map.get(r['experiment'], 'black')
        ax.scatter(r['throughput_u128_tok_s'], r[qmetric], s=140, color=c,
                   edgecolor='black', linewidth=0.7,
                   label=EXP_SHORT.get(r['experiment'], r['experiment']))
        ax.annotate(EXP_SHORT.get(r['experiment'], r['experiment']),
                    (r['throughput_u128_tok_s'], r[qmetric]),
                    textcoords='offset points', xytext=(8, 4), fontsize=8)
    # Baseline reference line
    base_q = perf_summary[perf_summary['experiment'] == BASELINE_EXP][qmetric].iloc[0]
    ax.axhline(base_q, color='gray', linestyle=':', linewidth=1, label='FP8 baseline')
    ax.set_xlabel('Output token throughput @ 128 users (tok/s)  ↑ higher is better')
    ax.set_ylabel(qlabel + '  ↑ higher is better')
    ax.set_title(f'Quality vs Throughput — {qlabel}', fontweight='bold')
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')

plt.suptitle('Quality–Performance Trade-off', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_quality_vs_throughput.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Quality vs P95 TTFT scatter (latency view) ────────────────────────────
tt_u128 = pareto[pareto['users'] == 128][['experiment', 'ttft_p95']]
perf_lat = perf_summary.merge(tt_u128, on='experiment', how='left', suffixes=('', '_u128'))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, qmetric, qlabel, ylim in zip(
    axes,
    ['arc_acc', 'gsm8k_strict'],
    ['ARC-Challenge acc', 'GSM8K exact-match (strict)'],
    [(0.55, 0.70), (0.80, 0.92)],
):
    for _, r in perf_lat.iterrows():
        if pd.isna(r['ttft_p95_u128']) or pd.isna(r[qmetric]):
            continue
        c = color_map.get(r['experiment'], 'black')
        ax.scatter(r['ttft_p95_u128'], r[qmetric], s=140, color=c,
                   edgecolor='black', linewidth=0.7,
                   label=EXP_SHORT.get(r['experiment'], r['experiment']))
        ax.annotate(EXP_SHORT.get(r['experiment'], r['experiment']),
                    (r['ttft_p95_u128'], r[qmetric]),
                    textcoords='offset points', xytext=(8, 4), fontsize=8)
    base_q = perf_summary[perf_summary['experiment'] == BASELINE_EXP][qmetric].iloc[0]
    ax.axhline(base_q, color='gray', linestyle=':', linewidth=1, label='FP8 baseline')
    ax.set_xlabel('P95 TTFT @ 128 users (ms)  ← lower is better')
    ax.set_ylabel(qlabel + '  ↑ higher is better')
    ax.set_title(f'Quality vs P95 TTFT — {qlabel}', fontweight='bold')
    ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best')

plt.suptitle('Quality–Latency Trade-off', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'turboquant_quality_vs_ttft.png', bbox_inches='tight')
plt.show()

## 11. Error analysis

In [ ]:
errors = df_all[(df_all['success'] == False) & (df_all['experiment'].isin(EXP_ORDER))]
print(f'Total failed requests: {len(errors)}')
if not errors.empty:
    display(errors.groupby(['run', 'experiment', 'users', 'error_type']).size()
                  .rename('count').reset_index())
else:
    print('✅ No errors across any experiment.')

## 12. Export unified summary

In [ ]:
unified = summary_agg.merge(
    server_summary,
    on=['experiment', 'users'],
    how='left',
)
unified['experiment'] = pd.Categorical(unified['experiment'], categories=EXP_ORDER, ordered=True)
unified = unified.sort_values(['experiment', 'users']).reset_index(drop=True)

out_path = RESULTS_ROOT / 'unified_summary_turboquant.csv'
unified.to_csv(out_path, index=False)
print(f'✅ Unified summary saved to: {out_path}')

perf_summary.to_csv(RESULTS_ROOT / 'turboquant_perf_quality_summary.csv', index=False)
print(f'✅ Per-experiment perf×quality summary saved to: {RESULTS_ROOT / "turboquant_perf_quality_summary.csv"}')

display(unified)